# Concurrency: threads and asyncio

**Stage:** Advanced · **Suggested time:** 45–90 minutes (capstone: several sessions)

**Prerequisites:** Concepts from notebooks 00–17. No variables or files from other notebooks are required.

## What you will learn

- Distinguish concurrency from parallelism
- Use a thread pool for waiting tasks
- Run bounded asynchronous work in Jupyter


## Your learning loop

1. Read the explanation and predict the next output.
2. Run the example with **Shift+Enter**.
3. Follow **Try a change**; explain the result in your own words.
4. Complete both exercises and enable their checks.
5. Build the mini-project, then use **Restart Kernel and Run All**.

Examples run as provided. Exercise starters are deliberately unfinished; their checks are disabled until you enable them. A successful Run All with checks disabled does **not** mean the exercises are complete.

Save a copy before extensive experiments. Use one notebook at a time to avoid mixing kernel state.


## 1. Overlap waiting with threads

Concurrency lets tasks make progress during each other's waits. This mock I/O uses sleep instead of external services. Thread pools often help blocking I/O. In a typical GIL-enabled CPython build, threads do not normally speed up CPU-bound Python loops; processes are one option for those workloads.


In [1]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch_mock(item):
    time.sleep(0.03)
    return f"item-{item}"

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(fetch_mock, range(6)))
print(results)
print(f"Elapsed: {time.perf_counter() - started:.3f}s")
assert results == [f"item-{n}" for n in range(6)]


['item-0', 'item-1', 'item-2', 'item-3', 'item-4', 'item-5']
Elapsed: 0.069s


**Try a change:** Compare max_workers=1 and 3. Timing varies, so do not assert an exact speedup.


## 2. Async functions cooperate

An async function returns a coroutine. await pauses it until an awaitable completes, allowing other work to run. Jupyter supports top-level await because it already runs an event loop; use asyncio.run(main()) at the entry point of a normal script instead.


In [2]:
import asyncio

async def fetch_async(item):
    await asyncio.sleep(0.03)
    return item * 2

async_results = await asyncio.gather(*(fetch_async(n) for n in range(4)))
print(async_results)
assert async_results == [0, 2, 4, 6]


[0, 2, 4, 6]


**Try a change:** Replace gather with a loop that awaits one task at a time and compare elapsed time.


## 3. Bound concurrency and timeouts

A semaphore limits simultaneous work. Use timeouts so stalled operations cannot wait indefinitely. Cleanup belongs in finally or context managers; cancellation is part of a task's lifecycle.


In [3]:
async def run_bounded():
    semaphore = asyncio.Semaphore(2)

    async def limited(item):
        async with semaphore:
            return await fetch_async(item)

    return await asyncio.wait_for(
        asyncio.gather(*(limited(n) for n in range(5))), timeout=2
    )

bounded_results = await run_bounded()
print(bounded_results)
assert bounded_results == [0, 2, 4, 6, 8]


[0, 2, 4, 6, 8]


**Try a change:** Set the limit to 1, then 5. Explain why the output order remains unchanged.


## Practice: solve before looking at the answers

Replace the starter code with your implementation. Run it, then change that exercise's `exercise_N_ready` flag to `True` and run its check cell. A failed assertion is feedback: compare the expected value with your result. Add at least one test of your own.


### Exercise 1: Threaded transformation

Implement threaded_lengths(words) with ThreadPoolExecutor.map and return lengths in input order. This is API practice, not a speed optimization for such a small CPU task.

<details>
<summary>Show a hint</summary>

map preserves the order of inputs.

</details>


In [4]:
def threaded_lengths(words):
    pass


In [5]:
exercise_1_ready = False  # Set True when your implementation is ready.
if exercise_1_ready:
    assert threaded_lengths(["a", "python", "code"]) == [1, 6, 4]
    assert threaded_lengths([]) == []
    print("Exercise 1: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.")


Exercise 1: not checked yet. Complete the code, then set exercise_1_ready = True.


### Exercise 2: An async function

Implement async_square(n): await asyncio.sleep(0), then return n*n. sleep(0) explicitly yields control without a positive delay.

<details>
<summary>Show a hint</summary>

Use async def and await inside it.

</details>


In [6]:
async def async_square(n):
    pass


In [7]:
exercise_2_ready = False  # Set True when your implementation is ready.
if exercise_2_ready:
    assert await async_square(3) == 9
    assert await asyncio.gather(async_square(0), async_square(-2)) == [0, 4]
    print("Exercise 2: supplied checks passed. Add your own edge cases!")
else:
    print("Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.")


Exercise 2: not checked yet. Complete the code, then set exercise_2_ready = True.


## Build it yourself

Simulate ten downloads with a concurrency limit of three, record their start and finish events, and verify that no more than three are active at once. Add a failing operation and decide how callers should receive its error.

This open-ended extension has no single supplied solution. Record your design choices and test your result.


In [8]:
# Your mini-project workspace. Add more cells as needed.


## Common mistakes and limits

time.sleep blocks the event-loop thread; asyncio.sleep yields control. Shared mutable state needs coordination. For process pools on Windows, put worker functions in an importable .py file and protect script startup with a __name__ guard.


## Before you move on

- [ ] I can explain the examples without reading the code.
- [ ] I changed at least two inputs and predicted the results.
- [ ] I completed both exercises and enabled their checks.
- [ ] I added a boundary or failure case of my own.
- [ ] I attempted the mini-project and recorded one lesson learned.

**Reflection:** What surprised you? What would you try next? Edit this Markdown cell with your notes.


[Course guide](../README.md) · [Worked solutions](../solutions/18_concurrency_threads_and_async_solutions.ipynb) · [Previous notebook](17_http_apis_and_json.ipynb) · [Next notebook](19_numpy_numerical_python.ipynb)
